In [18]:
from langgraph.graph import StateGraph , START , END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Annotated
from pydantic import BaseModel,Field
import operator

In [2]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [6]:
class EvalutionSchema(BaseModel):

    feedback: str = Field(description="Detaile feedback on the essay")
    score: int = Field(description="Score of the essay out of 10")

In [8]:
Structure_model  = model.with_structured_output(EvalutionSchema)


In [34]:
eassy = """India is a country and it is a very big country. There are many people in India and they live in many places. Some places are cities and some places are villages. I think India is good because it has many things and those things are very good.

India have many festivals. People celebrate festivals and they feel happy. There are different festivals at different times. Diwali is one festival and there are other festivals also. Festivals are nice because people enjoy them. I like festivals because they are good and people like good things very much.

Food is also there in India. People need food every day because food is important for eating. There are many types of food and different people eat different foods. Some food is spicy and some is not spicy. Indian food is famous and I think it tastes very nice.

India has schools and roads and buildings. Schools are for studying and roads are for going somewhere. Buildings are also useful because people stay inside them. Education is important and students should study because studying is important. There are many students and teachers who teach students in the schools.

India also has problems. Problems are not good and we should solve problems. Pollution is a problem and other problems are there too. People should do something about these problems. If everybody does good things, things will become good. The government should help people and people should help the government.

I want to say that India is a good country. It was good before and it can be good later also. We should love India because it is our country. Everybody should work hard and be nice. This is what I know about India and this is my essay. I have nothing more to write."""

In [35]:
propmt = f"""
Evaluate the follow
ing essay and provide detailed feedback and a score out of 10.{eassy}"""
Structure_model.invoke(propmt)

EvalutionSchema(feedback="The essay demonstrates a basic understanding of simple paragraph structure and covers multiple topics related to India, such as culture, food, infrastructure, and societal challenges. However, the writing is extremely repetitive, overly simplistic, and lacks vocabulary variety. Concepts are stated with redundant phrasing (e.g., 'food is important for eating', 'studying is important'). Additionally, there are minor grammatical issues such as subject-verb disagreement ('India have'). To improve, the author should use more varied sentence structures, expand on specific details rather than stating generalities, incorporate richer vocabulary, and eliminate tautological expressions.", score=3)

In [36]:
class UPES(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [37]:
def evaluate_language(state: UPES) -> UPES:
    prompt = f"Evaluate the language of the following essay and provide detailed feedback and a score out of 10. Essay: {state['essay']}"
    result = Structure_model.invoke(prompt)
 
    return {'language_feedback': result.feedback, 'individual_scores': [result.score]}

In [38]:
def evaluate_analysis(state: UPES) -> UPES:
    prompt = f"Evaluate the analysis of the following essay and provide detailed feedback and a score out of 10. Essay: {state['essay']}"
    result = Structure_model.invoke(prompt)
 
    return {'analysis_feedback': result.feedback, 'individual_scores': [result.score]}

In [39]:
def evaluate_clarity(state: UPES) -> UPES:
    prompt = f"Evaluate the clarity of the following essay and provide detailed feedback and a score out of 10. Essay: {state['essay']}"
    result = Structure_model.invoke(prompt)
 
    return {'clarity_feedback': result.feedback, 'individual_scores': [result.score]}

In [40]:
def final_evaluation(state: UPES) -> UPES:
    prompt = f"Provide an overall evaluation of the following essay based on the language, analysis, and clarity feedback. Essay: {state['essay']}. Language Feedback: {state['language_feedback']}. Analysis Feedback: {state['analysis_feedback']}. Clarity Feedback: {state['clarity_feedback']}."
    result = Structure_model.invoke(prompt)
    score = sum(state['individual_scores']) / len(state['individual_scores']) 
    return {'overall_feedback': result.feedback, 'avg_score': score}

In [41]:
graph = StateGraph(UPES)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START , 'evaluate_language')
graph.add_edge(START , 'evaluate_analysis')
graph.add_edge(START , 'evaluate_clarity')
graph.add_edge('evaluate_language' , 'final_evaluation')
graph.add_edge('evaluate_analysis' , 'final_evaluation')
graph.add_edge('evaluate_clarity' , 'final_evaluation')

graph.add_edge('final_evaluation' , END)

workflow = graph.compile()

In [42]:
propmt = {
    'essay': eassy,
}

In [44]:
workflow.invoke(propmt)


GoogleAPIError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}